# Experiment comparison with MLflow Search API, parameter plots, and run comparison

Comparing training runs by hand — writing down accuracies in a spreadsheet — doesn't scale past two or three attempts. MLflow's Search API lets you query logged runs programmatically, plot parameter-to-metric relationships, and pick the best configuration without guesswork.

last_verified: 2026-07-18 · MLflow n/a

## Setup

Local SQLite tracking so nothing leaves this notebook. I'll use the Iris dataset — small,多-class, easy to iterate on.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

mlflow.set_tracking_uri("sqlite:///mlflow_comparison.db")

In [ ]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Create an experiment

Grouping runs under a named experiment keeps the search focused later.

In [ ]:
experiment_name = "iris-classifier-grid"
exp = mlflow.set_experiment(experiment_name)
print(f"Experiment ID: {exp.experiment_id}")

## Run a grid of classifiers

I'll train four different model types with varying hyperparameters so there's enough data to make the comparison meaningful.

In [ ]:
configs = [
    {"model_type": "RandomForest", "n_estimators": 50, "max_depth": 5},
    {"model_type": "RandomForest", "n_estimators": 100, "max_depth": 10},
    {"model_type": "RandomForest", "n_estimators": 200, "max_depth": None},
    {"model_type": "GradientBoosting", "n_estimators": 50, "learning_rate": 0.1},
    {"model_type": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.05},
    {"model_type": "GradientBoosting", "n_estimators": 200, "learning_rate": 0.01},
    {"model_type": "ExtraTrees", "n_estimators": 100, "max_depth": None},
    {"model_type": "SVC", "C": 1.0, "gamma": "scale"},
    {"model_type": "SVC", "C": 10.0, "gamma": "auto"},
]

model_map = {
    "RandomForest": lambda p: RandomForestClassifier(
        n_estimators=p["n_estimators"], max_depth=p.get("max_depth"), random_state=42
    ),
    "GradientBoosting": lambda p: GradientBoostingClassifier(
        n_estimators=p["n_estimators"],
        learning_rate=p["learning_rate"],
        random_state=42,
    ),
    "ExtraTrees": lambda p: ExtraTreesClassifier(
        n_estimators=p["n_estimators"], max_depth=p.get("max_depth"), random_state=42
    ),
    "SVC": lambda p: SVC(C=p["C"], gamma=p["gamma"], random_state=42),
}

run_ids = []

for cfg in configs:
    with mlflow.start_run() as run:
        run_ids.append(run.info.run_id)
        for k, v in cfg.items():
            mlflow.log_param(k, v)

        model = model_map[cfg["model_type"]](cfg)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")
        cv_scores = cross_val_score(model, X_train, y_train, cv=3)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_weighted", f1)
        mlflow.log_metric("cv_mean", cv_scores.mean())
        mlflow.log_metric("cv_std", cv_scores.std())

        mlflow.sklearn.log_model(model, "model")
        print(f"{cfg['model_type']:20s} accuracy={acc:.4f}  f1={f1:.4f}")

Nine runs logged. Now the interesting part — extracting patterns from those numbers.

## MLflow Search API

`mlflow.search_runs()` returns a DataFrame of all runs in an experiment. You can filter, sort, and select specific columns.

In [ ]:
runs_df = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.accuracy DESC"],
)

display_cols = [
    "run_id",
    "params.model_type",
    "params.n_estimators",
    "params.learning_rate",
    "params.C",
    "params.gamma",
    "metrics.accuracy",
    "metrics.f1_weighted",
    "metrics.cv_mean",
]

# Only keep columns that exist
available = [c for c in display_cols if c in runs_df.columns]
runs_df[available]

### Filter with search_string

The `search_string` parameter accepts MLflow Search Expression syntax — useful when you have dozens or hundreds of runs.

In [ ]:
# Only tree-based models with accuracy > 0.95
filtered = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="params.model_type != 'SVC' and metrics.accuracy > 0.95",
    order_by=["metrics.accuracy DESC"],
)

filtered[available]

### Get the best run programmatically

Once the DataFrame is sorted, the top row is the best run by the chosen metric.

In [ ]:
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_model_type = best_run["params.model_type"]
best_accuracy = best_run["metrics.accuracy"]
print(f"Best: {best_model_type} (run {best_run_id[:8]}) — accuracy {best_accuracy:.4f}")

## Parameter plots

Visualizing how parameters affect metrics reveals patterns that a table hides — for example, whether more estimators always helps or if there's a plateau.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: n_estimators vs accuracy for tree-based models
tree_runs = runs_df[
    runs_df["params.model_type"].isin(["RandomForest", "ExtraTrees"])
].copy()

if not tree_runs.empty:
    tree_runs["n_estimators"] = pd.to_numeric(tree_runs["params.n_estimators"])
    for m_type in tree_runs["params.model_type"].unique():
        subset = tree_runs[tree_runs["params.model_type"] == m_type]
        axes[0].plot(
            subset["n_estimators"],
            subset["metrics.accuracy"],
            marker="o",
            label=m_type,
        )
    axes[0].set_xlabel("n_estimators")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Tree models: estimators vs accuracy")
    axes[0].legend()

# Plot 2: model_type comparison box
model_order = runs_df.groupby("params.model_type")["metrics.accuracy"].mean().sort_values(
    ascending=False
).index.tolist()
sns.barplot(
    data=runs_df,
    x="params.model_type",
    y="metrics.accuracy",
    order=model_order,
    ax=axes[1],
)
axes[1].set_xlabel("Model type")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy by model type")

plt.tight_layout()
plt.show()

### Correlation: parameter vs metric

For the GradientBoosting runs, I can check whether lower learning rates correlate with better generalization.

In [ ]:
gb_runs = runs_df[runs_df["params.model_type"] == "GradientBoosting"].copy()

if not gb_runs.empty:
    gb_runs["learning_rate"] = pd.to_numeric(gb_runs["params.learning_rate"])
    gb_runs["n_estimators"] = pd.to_numeric(gb_runs["params.n_estimators"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].scatter(gb_runs["learning_rate"], gb_runs["metrics.accuracy"], s=80)
    axes[0].set_xlabel("Learning rate")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("GBM: learning_rate vs accuracy")

    axes[1].scatter(gb_runs["n_estimators"], gb_runs["metrics.accuracy"], s=80)
    axes[1].set_xlabel("n_estimators")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("GBM: n_estimators vs accuracy")

    plt.tight_layout()
    plt.show()
else:
    print("No GradientBoosting runs to plot")

## Run comparison with MlflowClient

The `MlflowClient` gives lower-level access — compare two specific runs side by side.

In [ ]:
client = mlflow.MlflowClient()

# Compare best run vs second-best
if len(runs_df) >= 2:
    top2_ids = runs_df["run_id"].iloc[:2].tolist()

    comparison = []
    for rid in top2_ids:
        run_data = client.get_run(rid).data
        row = {"run_id": rid[:8]}
        row.update({f"param_{k}": v for k, v in run_data.params.items()})
        row.update({f"metric_{k}": float(v) for k, v in run_data.metrics.items()})
        comparison.append(row)

    comparison_df = pd.DataFrame(comparison).set_index("run_id")
    comparison_df

### CV stability check

A model with high accuracy but high cv_std is fragile — small data changes produce different results. The ratio `cv_mean / cv_std` gives a quick stability signal.

In [ ]:
runs_df["stability_ratio"] = (
    runs_df["metrics.cv_mean"] / runs_df["metrics.cv_std"].replace(0, np.nan)
)

fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    runs_df["metrics.accuracy"],
    runs_df["stability_ratio"],
    c=range(len(runs_df)),
    s=100,
    cmap="viridis",
)

for _, row in runs_df.iterrows():
    ax.annotate(
        row["params.model_type"][:4],
        (row["metrics.accuracy"], row["stability_ratio"]),
        fontsize=8,
    )

ax.set_xlabel("Test accuracy")
ax.set_ylabel("CV mean / CV std (stability)")
ax.set_title("Accuracy vs cross-validation stability")
plt.tight_layout()
plt.show()

## Register the best stable model

I'll pick the run with the best cv_mean that also has cv_std below a threshold — accuracy is not enough if it's not reliable.

In [ ]:
# Filter to reasonably stable runs, then pick best by cv_mean
stable = runs_df[runs_df["metrics.cv_std"] < 0.05].copy()

if not stable.empty:
    best_stable = stable.sort_values("metrics.cv_mean", ascending=False).iloc[0]
    best_stable_id = best_stable["run_id"]

    model_uri = f"runs:/{best_stable_id}/model"
    registered_name = "iris-classifier-stable"

    result = mlflow.register_model(model_uri, registered_name)
    print(f"Registered {registered_name} v{result.version} "
          f"(cv_mean={best_stable['metrics.cv_mean']:.4f}, "
          f"cv_std={best_stable['metrics.cv_std']:.4f})")

    client.set_model_version_tag(
        name=registered_name,
        version=result.version,
        key="selection_criteria",
        value="best_cv_mean_with_cv_std_below_0.05",
    )
else:
    print("No stable runs found (all cv_std >= 0.05)")

## What I'd try next

- A proper hyperparameter sweep with `mlflow.search_runs()` feeding into Optuna or scikit-learn's `GridSearchCV` with MLflow logging built in
- Parallel coordinates plots to compare multi-dimensional parameter spaces across 20+ runs
- Setting up the MLflow UI (`mlflow ui`) to explore these plots interactively instead of in-notebook matplotlib
- Using the Search API in a CI pipeline to auto-promote the best run to Staging

## Got stuck on

- `search_runs()` returns columns named `params.X` and `metrics.Y` with string values — numeric params need `pd.to_numeric()` before plotting
- The `filter_string` syntax uses single-word operators (`!=`, `>`, `<`) but field names must be prefixed with `params.` or `metrics.` — I kept forgetting the prefix
- `MlflowClient.get_run()` returns `run.data.params` as a dict of strings even for numeric values — explicit float conversion is needed
- Cross-validation metrics logged as separate runs don't automatically correlate — you have to pair them manually in analysis